<a href="https://colab.research.google.com/github/Sai875/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — ML Task Framing

**Provisional lane: Refresh / Content Opportunity Scoring**

My lane is to help an SEO/content editor decide which content pages to review first. The intended output is a ranked review queue, not an automatic instruction to edit a page. I will use the anonymized starter dataset and treat the current decline label as a proxy for framing; it is not a future outcome, so it cannot prove that a page will decline later or that editing it will improve traffic.

## 1. My lane as an ML task (type)

**Task type: Ranking / scoring.** The practical question is “Which pages should an editor review first?” A score can order pages by review priority, and the editor can inspect the highest-ranked pages first.

**Decision and action:** An SEO/content editor reviews a limited number of high-priority pages, checks the evidence, and decides whether an update is warranted.

**Cost of wrong calls:** False positives use review time on pages that may not need attention; false negatives leave potentially declining pages out of the review queue. The score is decision support, not a replacement for editorial judgment.

## 2. Target or proxy

For this first framing exercise, I will use `is_declining_proxy`, defined as 1 when the dataset's observed `trend_direction` is `down`, and 0 otherwise. This is a **proxy label**, not an ideal future-looking target: it summarizes the current window rather than measuring what happens after the prediction. It is useful for sketching how a review queue could be assessed, but later work should prefer a properly time-separated future outcome if the data supports it.

Because the proxy is derived from `trend_direction` (and related to `trend_pct`), those columns must not be used as input features when training a model to predict this proxy. That would leak the answer. Pseudonymous IDs are for identification/grouping only, not model features.

## 3. Success metric

**Primary framing metric: Precision@K**, initially using K = 50 if the data contains at least 50 rows. Precision@50 asks: among the 50 pages at the top of a ranking, what fraction have the proxy decline label? This matches the limited editorial-review capacity decision better than overall accuracy alone.

This metric evaluates alignment with the current proxy label, not whether edits cause recovery. I would compare any learned ranking with a transparent hand-written baseline and use a time-aware or client-held-out evaluation before making stronger claims.

## 4. The unit of analysis, as a real dataframe

**Unit of analysis: one pseudonymized content page/item per row.** The following cell loads the anonymized starter CSV directly from the official starter repository (the dataset is intentionally not copied into this assignment repository), prints its shape and available columns, and displays a lane-specific slice as a real pandas dataframe.

In [ ]:
import pandas as pd
import numpy as np

DATA_URL = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_URL)

print(f"Starter data shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
print("Available columns:", ", ".join(df.columns))

# Select fields relevant to editorial review, keeping only columns that exist in this dataset.
preferred_columns = [
    "content_id", "content_type", "content_age_days", "days_since_last_update",
    "impressions_90d", "clicks_90d", "avg_position", "ctr", "trend_direction", "trend_pct"
]
lane_columns = [c for c in preferred_columns if c in df.columns]
lane_df = df[lane_columns].copy()

print("\nUnit of analysis: one content page/item per row")
display(lane_df.head(10))
print("Lane slice shape:", lane_df.shape)

### Sketch the target/proxy column

The next cell creates the proxy from the observed `trend_direction` field if that field exists. This demonstrates the target's shape; it does not mean this is a future-outcome label. Check the printed class balance and treat it as a limitation.

In [ ]:
if "trend_direction" not in df.columns:
    raise KeyError("Expected column 'trend_direction' was not found. Inspect df.columns and update the framing.")

lane_df["is_declining_proxy"] = df["trend_direction"].astype("string").str.lower().eq("down").astype("int8")
print("Proxy label counts (0 = not down, 1 = down):")
display(lane_df["is_declining_proxy"].value_counts(dropna=False).rename_axis("is_declining_proxy").to_frame("rows"))
print("Proxy positive rate:", round(float(lane_df["is_declining_proxy"].mean()), 3))
display(lane_df.head(10))

### How Precision@K would be checked

At this framing stage, no learned model or prediction score has been created yet. The helper below defines the metric so a later ranking can be evaluated. The example ranking uses impressions as a simple illustrative baseline only; it is not claimed to be the best rule.

In [ ]:
def precision_at_k(scores, labels, k=50):
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels, dtype=int)
    if len(scores) != len(labels) or len(labels) == 0:
        raise ValueError("Scores and labels must have the same non-zero length.")
    k = min(int(k), len(labels))
    top_indices = np.argsort(-np.nan_to_num(scores, nan=-np.inf))[:k]
    return float(labels[top_indices].mean())

if "impressions_90d" in df.columns:
    baseline_scores = pd.to_numeric(df["impressions_90d"], errors="coerce").fillna(0)
    k = min(50, len(df))
    print(f"Illustrative impressions-only baseline Precision@{k}: {precision_at_k(baseline_scores, lane_df['is_declining_proxy'], k):.3f}")
    print("Interpretation: this is a baseline against the current proxy label, not evidence of future performance or causal impact.")
else:
    print("No impressions_90d column found; choose a baseline score after inspecting the available fields.")

## 5. Why ML might beat a fixed rule here

A fixed rule such as “review pages updated more than 180 days ago and with at least 500 impressions” is easy to explain, but it uses only a small number of manually chosen thresholds. Content age, update recency, impressions, clicks, position, and content type may interact, and a single cutoff may not prioritize pages consistently across these cases.

A model may learn useful combinations of these signals, but it does not automatically beat a rule. I would keep the hand-written rule as a baseline, evaluate rankings on held-out clients or a time-separated set, and keep a model only if it improves the chosen metric without leakage. The real action is a human-reviewed editorial queue. Results should be described as measured decision support, not proof that an edit will increase traffic or a prediction of Google's ranking algorithm.

## 6. Self-check

- [ ] Task type, decision, user/action, and error costs are explicit.
- [ ] Target/proxy and its limitations are explained.
- [ ] Success metric is named and tied to the review-capacity decision.
- [ ] Actual starter data is loaded and a real lane-specific dataframe is displayed.
- [ ] Unit of analysis is stated and the proxy target column is shown.
- [ ] Leakage risks are explained; IDs and target-derived fields are not proposed as features.
- [ ] The ML rationale compares against a fixed-rule baseline without claiming ML must win.
- [ ] All cells have been run top-to-bottom and outputs are visible in the saved notebook.

**Before submission:** In Colab choose **Runtime → Run all**, fix any errors, then **File → Save a copy in GitHub** and choose this repository and path `work/notebooks/w02_ml_task_framing.ipynb`. Confirm the committed GitHub notebook shows outputs, then submit `https://github.com/Sai875/flyrank-ml-internship` on the ML-03 assignment card.